# Week 2 — UNet from Scratch (Reference Solution)

> **Mentor reference.** Keep private until after the Week 2 deadline.

Builds a **UNet from scratch** and trains it as a denoising autoencoder: add Gaussian noise to an image, train the network to recover the clean image. This is pure architecture practice — no diffusion yet — but the UNet built here is the exact backbone reused (with timestep conditioning added) in Week 4.

**Runtime:** ~5 min on Colab GPU. Runtime → Change runtime type → T4 GPU.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1. Data

We use MNIST at native 28×28. Images are scaled to `[-1, 1]` — the standard range for generative models, and the same range we'll use for diffusion in later weeks.

In [ ]:
tf = transforms.Compose([
    transforms.ToTensor(),                 # [0,1]
    transforms.Normalize((0.5,), (0.5,)),  # -> [-1,1]
])
train = datasets.MNIST("./data", train=True, download=True, transform=tf)
loader = DataLoader(train, batch_size=128, shuffle=True)
print("images:", len(train))

## 2. UNet building blocks

Three reusable modules:
- **DoubleConv** — two conv layers with normalization and activation.
- **Down** — downsample (maxpool) then DoubleConv.
- **Up** — upsample, concatenate the skip connection, then DoubleConv.

The skip connections (concatenating encoder features into the decoder) are the defining feature of a UNet and are what we'll lean on for gradient flow.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1),
            nn.GroupNorm(8, cout) if cout % 8 == 0 else nn.GroupNorm(1, cout),
            nn.SiLU(),
            nn.Conv2d(cout, cout, 3, padding=1),
            nn.GroupNorm(8, cout) if cout % 8 == 0 else nn.GroupNorm(1, cout),
            nn.SiLU(),
        )
    def forward(self, x):
        return self.net(x)

class Down(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.conv = DoubleConv(cin, cout)
    def forward(self, x):
        return self.conv(self.pool(x))

class Up(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.conv = DoubleConv(cin, cout)   # cin includes the concatenated skip channels
    def forward(self, x, skip):
        x = self.up(x)
        # pad if odd sizes cause a 1px mismatch
        dy = skip.size(-2) - x.size(-2)
        dx = skip.size(-1) - x.size(-1)
        x = F.pad(x, [dx // 2, dx - dx // 2, dy // 2, dy - dy // 2])
        return self.conv(torch.cat([skip, x], dim=1))

## 3. Assemble the UNet

In [ ]:
class UNet(nn.Module):
    def __init__(self, ch=1, base=32):
        super().__init__()
        self.inc  = DoubleConv(ch, base)        # 28
        self.d1   = Down(base, base * 2)        # 14
        self.d2   = Down(base * 2, base * 4)    # 7
        self.u1   = Up(base * 4 + base * 2, base * 2)
        self.u2   = Up(base * 2 + base, base)
        self.outc = nn.Conv2d(base, ch, 1)
    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.d1(x1)
        x3 = self.d2(x2)
        x = self.u1(x3, x2)
        x = self.u2(x, x1)
        return self.outc(x)

model = UNet().to(device)
print(sum(p.numel() for p in model.parameters()), "parameters")
# shape sanity check
_t = torch.randn(2, 1, 28, 28, device=device)
assert model(_t).shape == _t.shape
print("shape check passed")

## 4. Train as a denoiser

Add fixed-strength Gaussian noise to each batch and ask the UNet to reconstruct the clean image. MSE loss between output and clean target.

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=2e-4)
NOISE_STD = 0.6
EPOCHS = 3
hist = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for x, _ in loader:
        x = x.to(device)
        noisy = x + NOISE_STD * torch.randn_like(x)
        pred = model(noisy)
        loss = F.mse_loss(pred, x)
        opt.zero_grad(); loss.backward(); opt.step()
        running += loss.item() * x.size(0)
    epoch_loss = running / len(train)
    hist.append(epoch_loss)
    print(f"epoch {epoch} | mse {epoch_loss:.4f}")

## 5. Visualize denoising

In [ ]:
import matplotlib.pyplot as plt
model.eval()
x, _ = next(iter(loader))
x = x[:8].to(device)
noisy = x + NOISE_STD * torch.randn_like(x)
with torch.no_grad():
    recon = model(noisy)

def to_img(t):
    return (t.cpu().clamp(-1, 1) + 1) / 2

fig, axes = plt.subplots(3, 8, figsize=(12, 4.5))
for i in range(8):
    axes[0, i].imshow(to_img(x[i, 0]), cmap="gray"); axes[0, i].axis("off")
    axes[1, i].imshow(to_img(noisy[i, 0]), cmap="gray"); axes[1, i].axis("off")
    axes[2, i].imshow(to_img(recon[i, 0]), cmap="gray"); axes[2, i].axis("off")
axes[0, 0].set_ylabel("clean"); axes[1, 0].set_ylabel("noisy"); axes[2, 0].set_ylabel("recon")
plt.suptitle("Row 1: clean   Row 2: noisy input   Row 3: UNet reconstruction")
plt.tight_layout(); plt.show()

## Self-check answers (for mentor)

1. **Why skip connections?** They route high-resolution detail from encoder to decoder and create short gradient paths, so fine structure isn't lost through the bottleneck and training is more stable.
2. **Role of the bottleneck?** Forces a compressed representation capturing the most important global information; the skips restore local detail on the way back up.
3. **Transposed conv vs bilinear upsample?** Transposed conv learns the upsampling filter (can cause checkerboard artifacts); bilinear upsample is fixed and artifact-free (used here), usually followed by a conv.
4. **Output 64×64, k=3, s=2, p=1 → output size?** `floor((64+2-3)/2)+1 = 32`.

## Common mentee mistakes
- Off-by-one channel/size mismatch when concatenating skips (handled here with `F.pad`).
- Output channels not matching input (must be 1→1 for MNIST).
- Bottleneck too small for the data.